In [6]:
'''# How to run this code in Google Colab or Local Machine: 
#Before running this code, make sure to install the required python and needed packages. 
# Since Iam used to Visual studio code, below instructions are for VSC but can be true fot other similar IDEs.
# If you don't want to install them in Local Machine then you use the colab directly. 
# Choose Runtime -> Python 3(ipykernel)
# If the above is not available, Select another kernel -> Colab -> Auto Conect or New Colab server if you do not have

'''

import sys
print("Python:", sys.version)
print("Executable:", sys.executable)
print("Runtime is running")

''' Here we are using the Gdrive to store the file and read it from there. 
If you want to use the local machine or colab folder, you can use the commented code below to read the file.
'''
# If the file is stored in the local machine or in Colab folder, you can use the following code to read the file.
# file_path = r"C:\Users\veera\AI\SampleFile\test-product-review.parquet"
# file_path = r"\sample_data\test-product-review.parquet"

from google.colab import drive
import os
drive.mount('/content/drive')
print("Drive mounted:", os.path.ismount('/content/drive'))

file_path = '/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet'
print("File exists:"+file_path+" -", os.path.isfile(file_path))

# Read the Parquet file with specific columns and filter for English language reviews
# For better understanding I have filtered the data for few columns English language reviews only. You can remove the filter if you want to read all the data.

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

table = pq.read_table(file_path, columns=['review_id','review_body', 'language','stars'],
                      filters=[[('language', '==', 'en')]])

#Print the first few rows of the DataFrame to see how the data looks like.
df = table.to_pandas()
df.info()
# print(df.head(4))
#print(df.shape)
# print("Count of each star rating:"+str(df["stars"].value_counts().to_dict()))


import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download these resources once
nltk.download("punkt")
nltk.download("punkt_tab")  # Needed by some NLTK versions
nltk.download("stopwords")
nltk.download("wordnet")

stop_words = set(stopwords.words("english"))
# Keep negation words because they can change review sentiment
stop_words -= { "no", "not", "never", "none", "neither", "nor", "noone", "nobody", "nothing", "nowhere", "hardly", "scarcely", "barely", "seldom", "rarely", "don't", "doesn't", "didn't", "isn't", "aren't", "wasn't", "weren't", "haven't", "hasn't", "hadn't", "won't", "wouldn't", "shouldn't", "can't", "cannot", "couldn't", }

lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def preprocess_review(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in stop_words]

    # Apply lemmatization, then stemming
    lemmas = [
        lemmatizer.lemmatize(
            lemmatizer.lemmatize(token, pos="v"),
            pos="n"
        )
        for token in tokens
    ]
    stems = [stemmer.stem(token) for token in lemmas]

    return tokens, lemmas, stems

df[["tokens", "lemmas", "stems"]] = df["review_body"].apply(
    lambda text: __import__("pandas").Series(preprocess_review(text))
)

# Text column to use later for vectorization/model training
df["review_body_processed"] = df["stems"].apply(" ".join)

# print(df[["review_body", "tokens", "lemmas", "stems"]].head())
# print(df[["review_body", "review_body_processed"]].head())
# df.info()


# 2) Data cleaning and label preparation

# df = df[["review_id", "review_body", "stars"]].dropna().copy()

# Convert stars to numeric
df["stars"] = pd.to_numeric(df["stars"], errors="coerce")
# df = df.dropna(subset=["stars"]).reset_index(drop=True)

# Keep only clear sentiment examples
def create_sentiment_label(stars):
    if stars >= 4:
        return "positive"
    elif stars <= 2:
        return "negative"
    else:
        return "neutral"

df["sentiment"] = df["stars"].apply(create_sentiment_label)

# # Remove neutral examples for a simpler binary sentiment task
# df = df[df["sentiment"] != "neutral"].reset_index(drop=True)

# print(df[["stars", "sentiment"]].head(10))
# print(df["stars"].value_counts())
# print(df["sentiment"].value_counts())
df.info()




Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Executable: /usr/bin/python3
Runtime is running
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted: True
File exists:/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet - True
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 460 entries, 0 to 459
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_id    460 non-null    object
 1   review_body  460 non-null    object
 2   language     460 non-null    object
 3   stars        460 non-null    int32 
dtypes: int32(1), object(3)
memory usage: 12.7+ KB


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 460 entries, 0 to 459
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   review_id              460 non-null    object
 1   review_body            460 non-null    object
 2   language               460 non-null    object
 3   stars                  460 non-null    int32 
 4   tokens                 460 non-null    object
 5   lemmas                 460 non-null    object
 6   stems                  460 non-null    object
 7   review_body_processed  460 non-null    object
 8   sentiment              460 non-null    object
dtypes: int32(1), object(8)
memory usage: 30.7+ KB


In [7]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# 1. Sample Dataset with 3 classes (positive, negative, neutral)
# data = {
#     "review_body_processed": [
#         "product absolute garbage stop working immediately",
#         "terrible experience would never buy again",
#         "not happy at all completely useless",
#         "awful quality very poor customer service",
#         "completely disappointed with purchase",
#         "product is okay nothing special average quality",
#         "it works fine not bad standard item",
#         "meets expectations nothing more nothing less",
#         "amazing quality fast delivery loved packaging",
#         "exceeded expectations works wonderfully",
#         "fantastic product highly recommend to everyone",
#         "very happy satisfied with buy absolute perfection",
#     ],
#     "sentiment": [
#         "negative",
#         "negative",
#         "negative",
#         "negative",
#         "negative",
#         "neutral",
#         "neutral",
#         "neutral",
#         "positive",
#         "positive",
#         "positive",
#         "positive",
#     ],
# }
# df = pd.DataFrame(data)

# 2. Train/Test Split (stratified across the 3 sentiment classes)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["review_body_processed"],
    df["sentiment"],
    test_size=0.3,
    random_state=42,
   stratify =df["sentiment"],
)

# 3. Feature Extraction (TF-IDF with unigrams and bigrams)
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

# 4. Define the Classical Models to Compare
models = {
    "Logistic Regression": LogisticRegression(),
    "Linear SVM (LinearSVC)": LinearSVC(random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(),
}


# 5. Helper Function to Calculate Multi-Class Metrics
def evaluate_metrics(y_true, y_pred):
  acc = accuracy_score(y_true, y_pred)
  # Using 'weighted' average since we now have 3 classes (positive, negative, neutral)
  prec, rec, f1, _ = precision_recall_fscore_support(
      y_true, y_pred, average="weighted", zero_division=0
  )
  return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1}


# 6. Train, Predict, and Evaluate Each Model
results = []
model_names = []

for name, model in models.items():
  # Train the model on string sentiments
  model.fit(X_train, y_train)

  # Generate predictions on the test set
  y_pred = model.predict(X_test)

  # Compute metrics and store results
  metrics = evaluate_metrics(y_test, y_pred)
  results.append(metrics)
  model_names.append(name)

# 7. Create a Summary Comparison DataFrame
comparison_df = pd.DataFrame(results, index=model_names)

print("Multi-Class Model Performance Comparison:")
print(comparison_df)

Multi-Class Model Performance Comparison:
                         Accuracy  Precision    Recall  F1-Score
Logistic Regression      0.659420   0.530477  0.659420  0.587958
Linear SVM (LinearSVC)   0.644928   0.596182  0.644928  0.599126
Multinomial Naive Bayes  0.608696   0.489887  0.608696  0.542813
